# Legendas automáticas: encoder ViT-B/16

Faculdade de Ciências e Tecnologias em Engenharia - Campus UnB Gama <br>
Tópicos Especiais em Matemática Aplicada: Deep Learning <br>
Professor: Vinicius Rispoli<br>
Alunos: Eliseu Kadesh, Pedro Barbosa e Raquel Eucaria.

Treina três decoders (LSTM, GRU e Transformer) sobre o mesmo encoder ViT-B/16 congelado e compara o BLEU. Contexto, resultados e limitações estão no [README](../README.md).

## 1. Configuração e dados

In [ ]:
!pip install torchmetrics
!pip install -q kagglehub

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.models as models
from torchvision.models import ViT_B_16_Weights
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pad_sequence
from torchmetrics.text import BLEUScore
from PIL import Image
import os
import pandas as pd
import kagglehub
import spacy
import time
import math

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Usando dispositivo: {device}")

# Baixar spacy se não existir
try:
    spacy_eng = spacy.load("en_core_web_sm")
except:
    print("⏳ Baixando modelo de linguagem spacy...")
    os.system("python -m spacy download en_core_web_sm")
    spacy_eng = spacy.load("en_core_web_sm")

✅ Usando dispositivo: cuda


### Vocabulário e dataset

Mesmo vocabulário e dataset do [notebook 01](01_encoder_resnet50.ipynb).

In [ ]:
class Vocabulary:
    def __init__(self, freq_threshold):
        self.itos = {0: "<PAD>", 1: "<SOS>", 2: "<EOS>", 3: "<UNK>"}
        self.stoi = {"<PAD>": 0, "<SOS>": 1, "<EOS>": 2, "<UNK>": 3}
        self.freq_threshold = freq_threshold

    def __len__(self):
        return len(self.itos)

    @staticmethod
    def tokenizer_eng(text):
        return [tok.text.lower() for tok in spacy_eng.tokenizer(str(text))]

    def build_vocabulary(self, sentence_list):
        frequencies = {}
        idx = 4
        for sentence in sentence_list:
            for word in self.tokenizer_eng(sentence):
                frequencies[word] = frequencies.get(word, 0) + 1
                if frequencies[word] == self.freq_threshold:
                    self.stoi[word] = idx
                    self.itos[idx] = word
                    idx += 1

    def numericalize(self, text):
        tokenized_text = self.tokenizer_eng(text)
        return [self.stoi.get(token, self.stoi["<UNK>"]) for token in tokenized_text]

class FlickrDataset(Dataset):
    def __init__(self, root_dir, captions_file, transform=None, freq_threshold=5):
        self.root_dir = root_dir
        self.df = pd.read_csv(captions_file)
        self.transform = transform
        self.imgs = self.df["image"]
        self.captions = self.df["caption"]
        self.vocab = Vocabulary(freq_threshold)
        self.vocab.build_vocabulary(self.captions.tolist())

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        caption = self.captions[index]
        img_id = self.imgs[index]
        img = Image.open(os.path.join(self.root_dir, img_id)).convert("RGB")

        if self.transform:
            img = self.transform(img)

        numericalized_caption = [self.vocab.stoi["<SOS>"]]
        numericalized_caption += self.vocab.numericalize(caption)
        numericalized_caption.append(self.vocab.stoi["<EOS>"])

        return img, torch.tensor(numericalized_caption)

class MyCollate:
    def __init__(self, pad_idx):
        self.pad_idx = pad_idx

    def __call__(self, batch):
        imgs = [item[0].unsqueeze(0) for item in batch]
        imgs = torch.cat(imgs, dim=0)
        targets = [item[1] for item in batch]
        targets = pad_sequence(targets, batch_first=True, padding_value=self.pad_idx)
        return imgs, targets

### Encoder e decoders

ViT-B/16 congelado, com a cabeça de classificação trocada por uma projeção treinável de 768 para 256 dimensões. Os decoders são os mesmos do notebook 01, com `hidden_size` 512 nas RNNs.

In [ ]:
class EncoderViT(nn.Module):
    def __init__(self, embed_size, train_backbone=False):
        super(EncoderViT, self).__init__()
        # Carrega o ViT pré-treinado
        self.vit = models.vit_b_16(weights=ViT_B_16_Weights.DEFAULT)

        # Substitui a cabeça de classificação para projetar no tamanho do embedding
        self.vit.heads = nn.Linear(768, embed_size)
        self.bn = nn.BatchNorm1d(embed_size, momentum=0.01)

        # Congela pesos para não treinar tudo do zero
        for name, param in self.vit.named_parameters():
            if "heads" in name:
                param.requires_grad = True
            else:
                param.requires_grad = train_backbone

    def forward(self, images):
        features = self.vit(images)
        features = self.bn(features)
        return features

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

class DecoderTransformer(nn.Module):
    def __init__(self, embed_size, vocab_size, num_heads, num_layers):
        super(DecoderTransformer, self).__init__()
        self.embed = nn.Embedding(vocab_size, embed_size)
        self.pos_encoder = PositionalEncoding(embed_size)
        decoder_layer = nn.TransformerDecoderLayer(d_model=embed_size, nhead=num_heads, batch_first=True)
        self.transformer_decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)
        self.linear = nn.Linear(embed_size, vocab_size)
        self.dropout = nn.Dropout(0.5)

    def generate_mask(self, sz):
        mask = (torch.triu(torch.ones(sz, sz)) == 1).transpose(0, 1)
        mask = mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, float(0.0))
        return mask.to(device)

    def forward(self, features, captions):
        captions_input = captions[:, :-1]
        embeddings = self.dropout(self.pos_encoder(self.embed(captions_input)))
        features = features.unsqueeze(1) # [Batch, 1, Embed]
        tgt_mask = self.generate_mask(embeddings.size(1))
        output = self.transformer_decoder(tgt=embeddings, memory=features, tgt_mask=tgt_mask)
        outputs = self.linear(output)
        return outputs

class DecoderRNN(nn.Module):
    def __init__(self, embed_size, hidden_size, vocab_size, num_layers, rnn_type="LSTM"):
        super(DecoderRNN, self).__init__()
        self.rnn_type = rnn_type
        self.embed = nn.Embedding(vocab_size, embed_size)

        # Escolha entre LSTM e GRU
        if rnn_type == "LSTM":
            # input_size é embed_size
            self.rnn = nn.LSTM(embed_size, hidden_size, num_layers, batch_first=True)
        else: # GRU
            # input_size é embed_size
            self.rnn = nn.GRU(embed_size, hidden_size, num_layers, batch_first=True)

        # A camada linear de saída deve usar hidden_size
        self.linear = nn.Linear(hidden_size, vocab_size)
        self.dropout = nn.Dropout(0.5)

    def forward(self, features, captions):
        # 1. Cria os Embeddings das legendas
        # input: <SOS> O gato ...
        # output: O gato ... <EOS>
        embeddings = self.dropout(self.embed(captions[:, :-1]))

        # 2. Concatena a Imagem como se fosse a "primeira palavra" da frase
        # inout: [IMAGEM, <SOS>, O, gato, ...]
        inputs = torch.cat((features.unsqueeze(1), embeddings), dim=1)

        # 3. Passa pela RNN
        hiddens, _ = self.rnn(inputs)

        # 4. Gera as previsões para cada passo de tempo
        outputs = self.linear(hiddens)

        # O RNN retorna uma sequência de comprimento L (Imagem + L-1 tokens).
        return outputs[:, 1:, :]

## 2. Treino

### Função de treino

Adam (lr 3e-4) com `ReduceLROnPlateau`, 10 épocas e batch 32. A projeção do encoder é compartilhada e continua treinando com cada decoder, na ordem Transformer, GRU e LSTM.

In [ ]:
def train_model(encoder, decoder, data_loader, criterion, optimizer, num_epochs, scheduler=None, model_name="modelo"):
    encoder.train()
    decoder.train()
    print(f"🚀 Iniciando Treino: {model_name}")

    for epoch in range(num_epochs):
        start_time = time.time()
        total_loss = 0

        for idx, (imgs, captions) in enumerate(data_loader):
            imgs = imgs.to(device)
            captions = captions.to(device)

            optimizer.zero_grad()

            # Forward
            features = encoder(imgs)
            outputs = decoder(features, captions)
            targets = captions[:, 1:]

            # Ajuste de segurança para tamanhos
            if outputs.shape[1] != targets.shape[1]:
                min_len = min(outputs.shape[1], targets.shape[1])
                outputs = outputs[:, :min_len, :]
                targets = targets[:, :min_len]

            loss = criterion(outputs.reshape(-1, outputs.shape[2]), targets.reshape(-1))
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

            if idx % 100 == 0:
                print(f"   Epoch [{epoch+1}/{num_epochs}], Step [{idx}/{len(data_loader)}], Loss: {loss.item():.4f}")

        avg_loss = total_loss / len(data_loader)

        # Scheduler Step
        if scheduler:
            scheduler.step(avg_loss)

        print(f"✅ Epoch {epoch+1} finalizada! Loss Média: {avg_loss:.4f} | Tempo: {time.time() - start_time:.0f}s")

    print(f"🏁 Treino de {model_name} concluído!")

### Dados, hiperparâmetros e encoder

O treino usa augmentation leve (recorte aleatório, espelhamento e variação de brilho e contraste).

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((232, 232)),
    transforms.RandomCrop((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

print("⏳ Baixando dataset...")
path = kagglehub.dataset_download("adityajn105/flickr8k")
image_folder = os.path.join(path, "Images")
caption_file = os.path.join(path, "captions.txt")

# Dataset e Loader
dataset = FlickrDataset(
    root_dir=image_folder,
    captions_file=caption_file,
    transform=train_transform,
    freq_threshold=5
)
pad_idx = dataset.vocab.stoi["<PAD>"]

train_loader = DataLoader(
    dataset=dataset,
    batch_size=32, # Batch seguro para memória
    num_workers=2,
    shuffle=True,
    collate_fn=MyCollate(pad_idx=pad_idx)
)

# Hiperparâmetros
embed_size = 256
vocab_size = len(dataset.vocab)
num_heads = 4
num_layers = 1
learning_rate = 3e-4
num_epochs = 10

hidden_size = 512

encoder_vit = EncoderViT(embed_size, train_backbone=False).to(device)

⏳ Baixando dataset...


### Transformer

In [ ]:
#=========================
#     TRANSFORMER
#=========================

print("🤖 Instanciando Modelos (ViT + Transformer)...")
decoder_trans = DecoderTransformer(embed_size, vocab_size, num_heads, num_layers).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=pad_idx)
params = list(decoder_trans.parameters()) + list(filter(lambda p: p.requires_grad, encoder_vit.parameters()))
optimizer = optim.Adam(params, lr=learning_rate)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=1, factor=0.5)

# Rodar Treino
train_model(encoder_vit, decoder_trans, train_loader, criterion, optimizer, num_epochs=num_epochs, scheduler=scheduler, model_name="ViT_Transformer")

🤖 Instanciando Modelos (ViT + Transformer)...
🚀 Iniciando Treino: ViT_Transformer
✅ Epoch 1 finalizada! Loss Média: 3.7175 | Tempo: 299s
✅ Epoch 2 finalizada! Loss Média: 3.0742 | Tempo: 293s
✅ Epoch 3 finalizada! Loss Média: 2.8635 | Tempo: 296s
✅ Epoch 4 finalizada! Loss Média: 2.7299 | Tempo: 295s
✅ Epoch 5 finalizada! Loss Média: 2.6321 | Tempo: 297s
✅ Epoch 6 finalizada! Loss Média: 2.5557 | Tempo: 296s
✅ Epoch 7 finalizada! Loss Média: 2.4930 | Tempo: 295s
✅ Epoch 8 finalizada! Loss Média: 2.4385 | Tempo: 297s
✅ Epoch 9 finalizada! Loss Média: 2.3928 | Tempo: 295s
✅ Epoch 10 finalizada! Loss Média: 2.3501 | Tempo: 296s
🏁 Treino de ViT_Transformer concluído!


### GRU

In [ ]:
#=========================
#     GRU
#=========================

decoder_gru = DecoderRNN(embed_size, hidden_size, vocab_size, num_layers, rnn_type="GRU").to(device)

criterion = nn.CrossEntropyLoss(ignore_index=pad_idx)
# O encoder_vit é o mesmo, mas agora os parâmetros do decoder_gru serão treinados
params = list(decoder_gru.parameters()) + list(filter(lambda p: p.requires_grad, encoder_vit.parameters()))
optimizer = optim.Adam(params, lr=learning_rate)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=1, factor=0.5)

# Rodar Treino
train_model(encoder_vit, decoder_gru, train_loader, criterion, optimizer, num_epochs=num_epochs, scheduler=scheduler, model_name="ViT_GRU")

🚀 Iniciando Treino: ViT_GRU
✅ Epoch 1 finalizada! Loss Média: 3.6368 | Tempo: 291s
✅ Epoch 2 finalizada! Loss Média: 2.8977 | Tempo: 295s
✅ Epoch 3 finalizada! Loss Média: 2.6580 | Tempo: 295s
✅ Epoch 4 finalizada! Loss Média: 2.5104 | Tempo: 295s
✅ Epoch 5 finalizada! Loss Média: 2.4030 | Tempo: 295s
✅ Epoch 6 finalizada! Loss Média: 2.3180 | Tempo: 295s
✅ Epoch 7 finalizada! Loss Média: 2.2462 | Tempo: 295s
✅ Epoch 8 finalizada! Loss Média: 2.1844 | Tempo: 295s
✅ Epoch 9 finalizada! Loss Média: 2.1303 | Tempo: 295s
✅ Epoch 10 finalizada! Loss Média: 2.0807 | Tempo: 295s
🏁 Treino de ViT_GRU concluído!


### LSTM

In [ ]:
#=========================
#         LSTM
#=========================

decoder_lstm = DecoderRNN(embed_size, hidden_size, vocab_size, num_layers, rnn_type="LSTM").to(device)

criterion = nn.CrossEntropyLoss(ignore_index=pad_idx)
# O encoder_vit é o mesmo, mas agora os parâmetros do decoder_lstm serão treinados
params = list(decoder_lstm.parameters()) + list(filter(lambda p: p.requires_grad, encoder_vit.parameters()))
optimizer = optim.Adam(params, lr=learning_rate)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=1, factor=0.5)

# Rodar Treino
train_model(encoder_vit, decoder_lstm, train_loader, criterion, optimizer, num_epochs=num_epochs, scheduler=scheduler, model_name="ViT_LSTM")

🚀 Iniciando Treino: ViT_LSTM
✅ Epoch 1 finalizada! Loss Média: 3.7599 | Tempo: 295s
✅ Epoch 2 finalizada! Loss Média: 3.0064 | Tempo: 295s
✅ Epoch 3 finalizada! Loss Média: 2.7416 | Tempo: 294s
✅ Epoch 4 finalizada! Loss Média: 2.5791 | Tempo: 295s
✅ Epoch 5 finalizada! Loss Média: 2.4612 | Tempo: 293s
✅ Epoch 6 finalizada! Loss Média: 2.3677 | Tempo: 294s
✅ Epoch 7 finalizada! Loss Média: 2.2912 | Tempo: 295s
✅ Epoch 8 finalizada! Loss Média: 2.2230 | Tempo: 295s
✅ Epoch 9 finalizada! Loss Média: 2.1638 | Tempo: 294s
✅ Epoch 10 finalizada! Loss Média: 2.1110 | Tempo: 293s
🏁 Treino de ViT_LSTM concluído!


## 3. Geração e avaliação

Geração gulosa (até 20 tokens) e BLEU-4 do TorchMetrics em 150 lotes (4.800 pares imagem-legenda) do `train_loader`, com uma legenda de referência por imagem. As limitações dessa avaliação estão no [README](../README.md).

In [ ]:
# Função para gerar legenda com LSTM ou GRU
def generate_caption_rnn(encoder, decoder, image, vocab, max_len=20):
    encoder.eval()
    decoder.eval()

    with torch.no_grad():
        # 1. Extrair features da imagem
        features = encoder(image.unsqueeze(0).to(device))

        # 2. Preparar estado inicial
        states = None

        # A entrada inicial é a imagem
        inputs = features.unsqueeze(1)

        caption = []

        for _ in range(max_len):
            # Passar pela RNN
            hiddens, states = decoder.rnn(inputs, states)

            # Prever a próxima palavra
            outputs = decoder.linear(hiddens.squeeze(1))
            predicted = outputs.argmax(1)

            if predicted.item() == vocab.stoi["<EOS>"]:
                break

            # Adiciona a palavra na lista
            caption.append(vocab.itos[predicted.item()])

            # A próxima entrada é a palavra que acabamos de prever
            inputs = decoder.embed(predicted).unsqueeze(1)

    return caption

# Função para gerar legenda com Transformer
def generate_caption_transformer(encoder, decoder, image, vocab, max_len=20):
    encoder.eval()
    decoder.eval()

    with torch.no_grad():
        features = encoder(image.unsqueeze(0).to(device)).unsqueeze(1) # Memory

        # Começa com <SOS>
        indices = [vocab.stoi["<SOS>"]]

        for _ in range(max_len):
            tgt = torch.tensor(indices).unsqueeze(0).to(device)

            # No Transformer, passamos a sequência inteira gerada até agora
            # Ele vai prever a próxima palavra baseada em tudo que viu
            tgt_mask = decoder.generate_mask(tgt.size(1))

            embeddings = decoder.dropout(decoder.pos_encoder(decoder.embed(tgt)))
            output = decoder.transformer_decoder(tgt=embeddings, memory=features, tgt_mask=tgt_mask)
            output = decoder.linear(output[:, -1, :])

            predicted = output.argmax(1).item()

            if predicted == vocab.stoi["<EOS>"]:
                break

            indices.append(predicted)

    return [vocab.itos[idx] for idx in indices[1:]]

In [ ]:
def evaluate_model(encoder, decoder, data_loader, model_type, num_batches=20):
    bleu = BLEUScore()
    encoder.eval()
    decoder.eval()

    predicted_sentences = []
    actual_sentences = []

    with torch.no_grad():
        for idx, (imgs, captions) in enumerate(data_loader):
            if idx >= num_batches: break

            imgs = imgs.to(device)

            for i in range(imgs.size(0)):
                # 1. Gerar legenda predita
                if model_type == "Transformer":
                    prediction = generate_caption_transformer(encoder, decoder, imgs[i], dataset.vocab)
                else:
                    prediction = generate_caption_rnn(encoder, decoder, imgs[i], dataset.vocab)
                pred_str = " ".join(prediction)
                predicted_sentences.append(pred_str)

                # 2. Pegar legenda real
                caption_indices = captions[i].tolist()
                # Remove tokens especiais (0,1,2,3)
                real_caption = [dataset.vocab.itos[idx] for idx in caption_indices if idx not in [0, 1, 2, 3]]
                actual_sentences.append([" ".join(real_caption)]) # BLEU espera lista de referências

    # Calcular
    score = bleu(predicted_sentences, actual_sentences)
    return score.item()

In [ ]:
num_batches_avaliacao = 150

# 1. Avaliar LSTM
score_lstm = evaluate_model(encoder_vit, decoder_lstm, train_loader, model_type="RNN", num_batches=num_batches_avaliacao)
print(f"📊 BLEU Score LSTM: {score_lstm * 100:.2f}%")

# 2. Avaliar GRU
score_gru = evaluate_model(encoder_vit, decoder_gru, train_loader, model_type="RNN", num_batches=num_batches_avaliacao)
print(f"📊 BLEU Score GRU: {score_gru * 100:.2f}%")

# 3. Avaliar Transformer
score_trans = evaluate_model(encoder_vit, decoder_trans, train_loader, model_type="Transformer", num_batches=num_batches_avaliacao)
print(f"📊 BLEU Score Transformer: {score_trans * 100:.2f}%")

📊 BLEU Score LSTM: 6.31%
📊 BLEU Score GRU: 1.40%
📊 BLEU Score Transformer: 7.36%


## Resultados

A comparação com o encoder ResNet-50 e a análise estão no [README](../README.md).